
# 23 怎样从 WT-plus 自动化记录找到下一步

> **本章要解决的问题**：自动化跑完一堆任务，有的成功、有的等待、有的失败。你怎样从记录里判断“下一步该补数据、修输入、补跑场景，还是等交易日”？

## 本章导读

第12章讲因子流水线，第14章讲回放，第15章讲失败诊断。这一章讲**读记录**：

- 记录的结构：名称、状态、数据、原因；
- 四类下一步：补数据、修输入、补跑场景、等日期；
- 状态机；
- 可复查的下一步。

### 学习目标

- 能读懂一条自动化记录的 name/status/data/reason。
- 能区分资料完成、候选、报告、回放。
- 能判断 waiting 和 failed，并按层排查。
- 能写一条可复查的下一步，并给验收指标。

### 先修

第12、14、15章。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="23 怎样从 WT-plus 自动化记录找到下一步" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>23 怎样从 WT-plus 自动化记录找到下一步｜知识图</title>
<desc>本章主线：读 name/status/data/reason，按层判断补数据、修输入、补跑场景还是等交易日。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a32" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao32" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag32" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">23</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">怎样从 WT-plus 自动化记录找到下一步</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：读 name/status/data/reason，按层判断补数据、修输入、补跑场景还是等交易日。</text>
<rect x="41" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="121" y="133.8" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">自动化记录</text>
<text x="121" y="153.9" font-size="10.5" fill="#6E7B8B" text-anchor="middle">name / status / data /</text>
<text x="121" y="168.9" font-size="10.5" fill="#6E7B8B" text-anchor="middle">reason</text>
<rect x="51" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="68" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输入</text>
<rect x="227" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="307" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">按 reason 分层定位</text>
<text x="307" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">资料 / 因子 / 报告 / 回放</text>
<rect x="237" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="254" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">处理</text>
<rect x="413" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="493" y="133.8" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">可复查的下一步</text>
<text x="493" y="153.9" font-size="10.5" fill="#6E7B8B" text-anchor="middle">补数据 / 修输入 / 补跑 /</text>
<text x="493" y="168.9" font-size="10.5" fill="#6E7B8B" text-anchor="middle">等日期</text>
<rect x="423" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="440" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输出</text>
<rect x="599" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="679" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">验收指标</text>
<text x="679" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">waiting 不是 failed</text>
<rect x="609" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="626" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">检查</text>
<path d="M 201 149 L 227 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a32)"/>
<path d="M 387 149 L 413 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a32)"/>
<path d="M 573 149 L 599 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a32)"/>
<rect x="41" y="232" width="718" height="152" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<rect x="58" y="246" width="4" height="11" fill="#E07B39" rx="1"/>
<text x="70" y="256" font-size="11.5" fill="#1B3A5C" font-weight="bold">自动化记录 → 下一步</text>
<rect x="58" y="288" width="310" height="22" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.2" rx="6"/>
<text x="68" y="303" font-size="8.8" fill="#1B3A5C">验证因子 · A</text>
<rect x="200" y="292" width="62" height="14" fill="#1B3A5C" rx="4"/>
<text x="231" y="302.5" font-size="7.6" fill="#FFFFFF" text-anchor="middle">completed</text>
<text x="272" y="303" font-size="7.8" fill="#6E7B8B">R-102 已出报告</text>
<rect x="58" y="314" width="310" height="22" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.2" rx="6"/>
<text x="68" y="329" font-size="8.8" fill="#1B3A5C">原生回放</text>
<rect x="200" y="318" width="62" height="14" fill="#6E7B8B" rx="4"/>
<text x="231" y="328.5" font-size="7.6" fill="#FFFFFF" text-anchor="middle">waiting</text>
<text x="272" y="329" font-size="7.8" fill="#6E7B8B">等匹配交易日</text>
<rect x="58" y="340" width="310" height="22" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.2" rx="6"/>
<text x="68" y="355" font-size="8.8" fill="#1B3A5C">同步资料</text>
<rect x="200" y="344" width="62" height="14" fill="#E07B39" rx="4"/>
<text x="231" y="354.5" font-size="7.6" fill="#FFFFFF" text-anchor="middle">failed</text>
<text x="272" y="355" font-size="7.8" fill="#6E7B8B">缺分钟成交量</text>
<path d="M 378 322 L 418 322" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag32)"/>
<rect x="432" y="288" width="118" height="28" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="491" y="305.3" font-size="9.5" fill="#1B3A5C" text-anchor="middle">补数据</text>
<rect x="568" y="288" width="118" height="28" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="627" y="305.3" font-size="9.5" fill="#1B3A5C" text-anchor="middle">修输入</text>
<rect x="432" y="326" width="118" height="28" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="491" y="343.3" font-size="9.5" fill="#1B3A5C" text-anchor="middle">补跑场景</text>
<rect x="568" y="326" width="118" height="28" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="627" y="343.3" font-size="9.5" fill="#1B3A5C" text-anchor="middle">等交易日</text>
<text x="432" y="280" font-size="8.6" fill="#6E7B8B">下一步四选一</text>
</svg>

> **图**：本章主线：读 name/status/data/reason，按层判断补数据、修输入、补跑场景还是等交易日。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：研究员、WT 运维、策略开发。
- **输入**：自动化记录（name/status/data/reason）。
- **输出**：诊断、下一步、验收指标。
- **在流程的哪一步**：日常运维与排错。
- **失败模式**：把 waiting 当 failed、不写 reason、资料完成当因子完成。
- **最小工作流**：读记录 → 按 reason 分层 → 定位输入/计算/回放 → 写下一步 → 验收。

---
## 1. 一条记录长什么样

| 字段 | 例子 | 用途 |
|---|---|---|
| name | 验证因子 · A | 哪一步 |
| status | completed/failed/waiting | 结果 |
| data | rows/version_id/report_id | 证据 |
| reason | 缺少分钟成交量 | 原因 |

**没有 reason 的失败，等于没记录。**

---

## 2. 三类记录

### 2.1 资料

`同步资料 Skill`：rows=2；只说明“有材料”，不代表因子。

### 2.2 因子

`验证因子 · A`：有 report_id；才算“有报告”。

### 2.3 回放

`WT-plus 原生回放`：completed_days=0、reason=等待匹配交易日；是“等待”，不是“失败”。

### 2.4 工具链记录

| 记录 | 看什么 | 下一步 |
|---|---|---|
| RD-Agent run | 候选表达式/轮次 | 看是否过 DSR/PBO |
| KunQuant batch | 输入长度/是否 8 的倍数 | 补齐/重算 |
| Alphalens report | IC/ICIR/分组/换手 | 进 DSR/PBO |
| DSR/PBO verdict | 校正后显著性/过拟合概率 | 拒绝/进 fincore |
| fincore report | 组合绩效/归因 | 进回放 |
| Great Expectations suite | 字段/时序/异常值 | 修数据 |
| MLflow run_id | 版本/参数/指标 | 复现 |
| WT-plus 回放 | completed_days/reason | 等日期/修配置 |

### 代码：分类计数



In [2]:

rows = [("P", "v1", "j1", "r1"), ("P", "v2", None, None)]
print("资料数", len({r[0] for r in rows}))
print("候选数", len({r[1] for r in rows}))
print("有报告数", sum(r[3] is not None for r in rows))


资料数 1
候选数 2
有报告数 1



**读数**：2 个候选、1 个报告；不能把“2 个候选”当“2 个已验证因子”。

---

## 3. 状态与下一步

### 3.1 等待 vs 失败

- waiting：条件没满足（交易日/数据未到）；
- failed：执行出错（字段/代码/权限）。

### 3.2 四类下一步

| 现象 | 下一步 |
|---|---|
| 缺字段 | 补数据/换定义 |
| 报告为空 | 查覆盖率/日期/表达式 |
| 场景缺失 | 补跑 base/stress |
| 回放 0 天 | 等完整交易日 |

### 代码：场景完整性



In [3]:

receipts = [("A", "D", "base", "ok"), ("A", "D", "base", "ok"),
            ("A", "D", "stress", "failed"), ("B", "D", "stress", "ok")]
state = {(v, d, s): result for v, d, s, result in receipts}
print("A 需要补的场景", [s for s in ["base", "stress"] if state.get(("A", "D", s)) != "ok"])


A 需要补的场景 ['stress']



**读数**：A 的 stress 失败，需要补跑；base 已 ok。

---

## 4. 写一条可复查的下一步

### 4.1 模板

> 因为【证据】，下一步【动作】，由【谁】在【何时】做，验收【指标】。

### 4.2 例子

“因为 A 的 stress 场景 failed（reason=缺少分钟成交量），下一步补 11 月 2 日分钟成交量并重跑 stress；验收：report_id 生成、覆盖率 ≥ 90%。”

### 代码：日志扫描



In [4]:

events = [
    {"name": "同步资料 Skill", "status": "completed", "data": {"rows": 2}},
    {"name": "读取历史数据", "status": "completed", "data": {"rows": 0}},
    {"name": "WT-plus 原生回放", "status": "waiting", "data": {"completed_days": 0}},
]
for e in events:
    hint = "查日期/品种" if e["name"] == "读取历史数据" and e["data"].get("rows") == 0 else e["status"]
    print(e["name"], "->", hint)


同步资料 Skill -> completed
读取历史数据 -> 查日期/品种
WT-plus 原生回放 -> waiting



**读数**：读取历史数据 rows=0，先查日期/品种/权限；回放 waiting，等交易日。两者都不是“策略无效”。

**常见错误**：

1. 把 waiting 当 failed。
2. 不写 reason。
3. 把“资料完成”当“因子完成”。

---

## 5. 案例：一次“失败”的回放记录

| 字段 | 值 |
|---|---|
| name | WT-plus 原生回放 |
| status | failed |
| data.completed_days | 0 |
| reason | 等待匹配交易日 |

这条记录状态是 failed，但 reason 是“等待匹配交易日”，本质是 waiting。下一步不是改策略，而是等交易日/补历史。

### 代码：按 reason 纠正状态



In [5]:

record = {"name": "WT-plus 原生回放", "status": "failed",
          "data": {"completed_days": 0}, "reason": "等待匹配交易日"}

if record["status"] == "failed" and "等待" in record["reason"]:
    print("实际是 waiting：等完整交易日，不是策略失败")
else:
    print("按 failed 处理：查输入/日志")


实际是 waiting：等完整交易日，不是策略失败



**读数**：状态字段和 reason 冲突时，以 reason/证据为准；不要因为 status=failed 就重写策略。

**常见错误**：

1. 只看 status，不看 reason。
2. 把“0 天”当“回放失败”。
3. 不区分“等日期”和“缺数据”。

---

## 6. 一张排查顺序表

| 现象 | 先查 | 再查 | 最后 |
|---|---|---|---|
| 资料 rows=0 | 抓取/权限 | 解析 | 重试 |
| 候选无法计算 | 字段/品种 | 表达式 | 窗口 |
| 报告为空 | 日期/覆盖率 | 标签 | 分组 |
| 回放 0 天 | 交易日历 | 数据 | 撮合 |
| 回放 failed | reason | 日志 | 代码 |
| 工具链 failed | MLflow run_id | 日志 | 重跑 |

**顺序是：输入 → 计算 → 验证 → 回放。** 不要一上来改策略。

**常见错误**：

1. 从策略开始查。
2. 不按层记录。
3. 不写验收指标。

---

## 本章小结

### 一张表

| 记录 | 说明 | 下一步 |
|---|---|---|
| 资料完成 | 有材料 | 准入/提案 |
| 候选 | 未验证 | 计算/验证 |
| 报告 | 有 report_id | 读报告 |
| 回放 waiting | 等日期 | 等 |
| 回放 failed | 出错 | 修输入 |

### 六句话

1. 记录要有 name/status/data/reason。
2. 资料完成 ≠ 因子完成。
3. waiting 不是 failed。
4. 先查输入，再查计算，再查验证。
5. 下一步要具体、可复查。
6. 缺数据不是策略无效。

---

## 本章习题

**1.** 一条记录至少要有哪四个字段？

**2.** 资料 rows=2 说明什么？

**3.** 回放 completed_days=0、reason=等待交易日，是失败吗？

**4.** 读取历史数据 rows=0，先查什么？

**5.** 写一条“补跑 stress”的下一步模板。

<details><summary>参考答案</summary>

**1.** name、status、data、reason。

**2.** 有 2 条材料；不代表因子/报告。

**3.** 不是；是 waiting。

**4.** 日期/品种/权限/字段。

**5.** 因为 stress failed（reason=…），补数据并重跑，验收 report_id/覆盖率。

</details>

---

## 参考资料

- 第12、14、15、37章。
- WonderTrader-plus: https://github.com/newbigdeng/WonderTrader-plus
- wtpy-plus: https://github.com/newbigdeng/wtpy-plus
